In [2]:
from google.colab import files

uploaded = files.upload()

Saving KYC_AML_Policy.pdf to KYC_AML_Policy.pdf


In [3]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 65.8 MB/s eta 0:00:00


In [4]:
import fitz

pdf = fitz.open("KYC_AML_Policy.pdf")

text = ""

for page in pdf:
    text += page.get_text()

print("Pages:", len(pdf))
print("Characters:", len(text))

Pages: 9
Characters: 23082


In [5]:
!pip install -q pymupdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
import os
print("Libraries Installed Successfully")

Libraries Installed Successfully


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

print("Total Chunks:", len(chunks))

Total Chunks: 60


In [9]:
print(chunks[0])

POLICY GUIDELINES ON KYC/AML/CFT (DOMESTIC BRANCHES) 
Dear Customer, 
 
RBI Master Direction - Know Your Customer (KYC) Direction, 2016 dated Feb 25, 
2016 and updated on January 04, 2024 mandates compliance of KYC Standards. 
Accordingly, each customer of bank is required to submit at-least one document for 
address proof as well as for identity proof out of set of Officially Valid Documents (OVDs) 
while establishing banking relations with the bank. 
 
KYC NORMS/GUIDELINES/DIRECTIONS


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print(chunk_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(60, 384)


In [11]:
import faiss
import numpy as np

chunk_embeddings = np.array(
    chunk_embeddings
).astype("float32")

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(chunk_embeddings)

print("Total Vectors:", index.ntotal)

Total Vectors: 60


In [12]:
def search_document(question, top_k=3):

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    print("Question:", question)

    for rank, idx in enumerate(indices[0], 1):

        print("\n")
        print("="*100)
        print(f"Result {rank}")
        print("="*100)

        print(chunks[idx])

In [13]:
search_document(
    "What is KYC?"
)

Question: What is KYC?


Result 1
while establishing banking relations with the bank. 
 
KYC NORMS/GUIDELINES/DIRECTIONS 
 
Know your customer is the most important aspect in terms of legislative 
requirements. It refers to our relationship with customer at the time of on-boarding. Once 
customer based relationship is established it becomes resource for future customer 
banking relationship.  
  
In general terms, KYC is: 
 
 Making every reasonable effort to determine the true identity and beneficial 
ownership of accounts


Result 2
ix. 
“IGA” means Inter Governmental Agreement between the Governments of India 
and the USA to improve international tax compliance and to implement FATCA of 
the USA. 
 
x. 
“KYC Templates” means templates prepared to facilitate collating and reporting 
the KYC data to the CKYCR, for individuals and legal entities. 
 
xi. 
“Non-face-to-face customers” means customers who open accounts without 
visiting the branch/offices of the bank or meeting the offic

In [14]:
search_document(
    "What is Customer Due Diligence?"
)

Question: What is Customer Due Diligence?


Result 1
monitored for, and has measures in place for compliance with client due diligence 
and record-keeping requirements in line with the requirements and obligations under 
the act;  
(iv) the third party is not based in a country or jurisdiction assessed as high risk;  
(v) the reporting entity is ultimately responsible for client due diligence and undertaking 
enhanced due diligence measures, as applicable; and


Result 2
in the transaction or activity, is acting.  
 
v. 
“Walk-in Customer” means a person who does not have an account-based 
relationship with the Bank, but undertakes transactions with the Bank. 
 
vi. 
“Customer Due Diligence (CDD)” means identifying and verifying the customer 
and the beneficial owner, using reliable and independent sources of identification. 
 
vii. 
“Customer identification” means undertaking the process of CDD. 
 
viii.


Result 3
For the purpose of verifying the identity of customers at the time of 

In [15]:
search_document(
    "When is PAN mandatory?"
)

Question: When is PAN mandatory?


Result 1
KYC/AML Policy  
 
 
 
 
 
Page 9 of 9 
 
 
PAN REQUIREMENT: 
 
1. Accounts/Transactions in which PAN OR equivalent e-document OR form 60/61 
required rule 114B of the income tax rules, 1962 has made it mandatory for the 
customer to quote the PAN for certain banking transactions, which have been 
detailed as below: 
 
(i) 
Placing a time deposit (i.e. a term deposit) EXCEEDING Rs.50,000/ or 
aggregating to more than Rs. five lakh during a financial year with the bank; 
(ii)


Result 2
not have any income chargeable to income tax, he/she shall quote the PAN of his/ 
her father or mother or guardian, as the case may be.  
 
3. A person who doesn’t have PAN have to file a declaration in form 60 giving the 
transaction details. 
 
4. Penalty for contravention : Section 272B defines the penalty for contravention of 
above rules : 
i. If a person who is required to collect PAN is unable to do so, assessing 
officer may impose penalty of Rs. 10,000

In [19]:
!pip install -q transformers sentencepiece

In [21]:
!pip install -q transformers sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = "What is KYC in banking?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

a credit rating agency


In [23]:
def search_document(question, top_k=3):

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [24]:
def rag_answer(question):

    retrieved_chunks = search_document(
        question,
        top_k=3
    )

    context = "\n\n".join(
        retrieved_chunks
    )

    prompt = f"""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [25]:
print(
    rag_answer(
        "What is KYC?"
    )
)

 Making every reasonable effort to determine the true identity and beneficial ownership of accounts


In [26]:
print(
    rag_answer(
        "What is a Politically Exposed Person?"
    )
)

individuals who are or have been entrusted with prominent public functions by a foreign country
